In [3]:
!pip install scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ------------------ --------------------- 3.7/8.0 MB 21.5 MB/s eta 0:00:01
   ---------------------------------------  7.9/8.0 MB 20.8 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 19.2 MB/s  0:00:00
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   -------------------------- ------------- 2/3 [scikit-learn]

In [4]:
import torch
import numpy as np
import scipy.io
# import yfinance as yf
from sklearn.linear_model import Ridge

In [5]:
# ── 1. Load Heston parameters from MATLAB ─────────────────────────────────────
mat   = scipy.io.loadmat('heston_params.mat')
Param = mat['Param'].flatten()
V0, ThetaV, Kappa, SigmaV, RhoSV = Param
r = -0.001


In [ ]:
# ── 2. Simulate ALL V(t) paths once (Monte Carlo) ─────────────────────────────
dt      = 1/252
T       = 3.0
n_steps = int(T / dt)
n_paths = 5000          # all paths generated once, reused everywhere

torch.manual_seed(42)

# Pre-generate ALL random shocks for the entire simulation upfront
# Shape: (n_steps, n_paths) — reused for mu fitting AND ANN training
dW_V_all = torch.randn(n_steps, n_paths) * (dt**0.5)  # variance shocks
dW_S_all = torch.randn(n_steps, n_paths) * (dt**0.5)  # price shocks

# Simulate all V(t) paths
V_paths = torch.zeros(n_steps + 1, n_paths)
V_paths[0] = V0

for t in range(n_steps):
    V_prev = V_paths[t]
    # Since we didn't check the Feller condition (2*Kappa*ThetaV>SigmaV**2), we clamp V to be non-negative to avoid numerical issues
    V_paths[t+1] = torch.clamp(
        V_prev + Kappa*(ThetaV - V_prev)*dt
               + SigmaV*torch.sqrt(V_prev.clamp(min=0))*dW_V_all[t], 
        min=1e-6
    )


# V_paths shape: (n_steps+1, n_paths)
# Every subsequent step just reads from this tensor — no re-simulation


In [ ]:
# ── 3. Estimate mu(V) = alpha + beta*V from TSLA historical data ───────────────
tsla       = yf.download('TSLA', start='2015-07-10', end='2015-09-30')
prices     = tsla['Close'].values
log_ret    = np.diff(np.log(prices))

window = 10
mu_roll, V_roll = [], []
for i in range(window, len(log_ret)):
    mu_roll.append(np.mean(log_ret[i-window:i]) * 252)
    V_roll.append(np.var(log_ret[i-window:i])   * 252)

mu_roll = np.array(mu_roll).reshape(-1, 1)
V_roll  = np.array(V_roll).reshape(-1, 1)

reg   = Ridge(alpha=0.1).fit(V_roll, mu_roll)  # Ridge for stability
alpha_mu = float(reg.intercept_)
beta_mu  = float(reg.coef_[0])

print(f"mu(V) = {alpha_mu:.4f} + {beta_mu:.4f} * V(t)")

In [ ]:
# ── 4. Compute mu(t) paths directly from V_paths ──────────────────────────────
# Shape: (n_steps+1, n_paths) — no re-simulation needed
mu_paths = alpha_mu + beta_mu * V_paths

In [ ]:
# ── 5. ANN Policy ─────────────────────────────────────────────────────────────
class PolicyNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(4, 128), torch.nn.Tanh(),
            torch.nn.Linear(128, 128), torch.nn.Tanh(),
            torch.nn.Linear(128, 1), torch.nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

policy    = PolicyNet()
optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)
gamma     = 3.0

In [ ]:
# ── 6. Training Loop — reads from pre-simulated paths, no re-simulation ────────
for epoch in range(2000):

    W_ann      = torch.ones(n_paths)
    W_analytic = torch.ones(n_paths)

    for t in range(n_steps):

        V_t  = V_paths[t]       # read from pre-simulated V tensor
        mu_t = mu_paths[t]      # read from pre-computed mu tensor
        dW_S = dW_S_all[t]      # read pre-generated shock

        # Analytic Merton pi
        with torch.no_grad():
            pi_analytic = ((mu_t - r) / (gamma * V_t)).clamp(0, 1)
            # 0.5*pi_analytic**2*V_t is the correction from Its's Lemma becuase we took the log of wealth/returns, not wealth itself
            W_analytic  = W_analytic * torch.exp(
                (r + pi_analytic*(mu_t - r) - 0.5*pi_analytic**2*V_t)*dt 
                + pi_analytic*torch.sqrt(V_t)*dW_S
            )

        # ANN pi
        state  = torch.stack([
            torch.full((n_paths,), t*dt),
            V_t,
            W_ann.detach(),
            mu_t
        ], dim=1)
        pi_ann = policy(state).squeeze()
        W_ann  = W_ann * torch.exp(
            (r + pi_ann*(mu_t - r) - 0.5*pi_ann**2*V_t)*dt
            + pi_ann*torch.sqrt(V_t)*dW_S
        )

    # Loss over all Monte Carlo paths
    loss = -torch.mean(W_ann**(1 - gamma) / (1 - gamma))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        U_ann      = torch.mean(W_ann**(1-gamma)/(1-gamma)).item()
        U_analytic = torch.mean(W_analytic**(1-gamma)/(1-gamma)).item()
        print(f"Epoch {epoch:4d} | ANN: {U_ann:.4f} | Analytic: {U_analytic:.4f} | Loss: {loss.item():.4f}")